# Run Neuron Proofreader — Train & Inference

Trains the `VisionHGAT` split-correction model with `neuron-proofreader`'s own
`FragmentsDatasetCollection` / `Trainer`, then runs `InferencePipeline` to score
proposals, merge accepted ones into a corrected skeleton, and compute before/after
metrics.

This is the **end-to-end workflow**. It does not depend on the exploratory
walkthrough in `explore_proofreader_pipeline.ipynb` — the training and inference
pipelines each build their own `ProposalGraph` and generate their own proposals
internally from `fragments_path` / `gt_path`.

> **Related:** For a step-by-step look at the pipeline internals (load fragments,
> generate proposals, load ground truth, extract features), see
> [`explore_proofreader_pipeline.ipynb`](explore_proofreader_pipeline.ipynb).

**Steps:**
1. Setup + paths/config
2. Train `VisionHGAT` (bounded by `max_epochs` / `max_steps`)
3. Run `InferencePipeline` and report before/after metrics

**Note:** Requires a GPU and access to the raw image volume. `max_steps` below is
set small for a quick smoke run — set it to `None` to train full epochs. To skip
training and reuse an existing checkpoint, set `external_model_path` in the
inference step.

## Setup

In [1]:
import json
import os

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../zihan_gcs_token.json"
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"

from neuron_proofreader.config import Config, MLConfig, ProposalGraphConfig

## Define Paths and Configuration

Same data sources as `load_skeletons.ipynb`.

In [2]:
# Brain and segmentation identifiers
brain_id = "794495"
segmentation_id = "raw.unet_449_splits_and_merges_900000"

# GCS paths
fragments_path = f"gs://allen-nd-goog/from_google/{brain_id}/whole_brain/{segmentation_id}/swcs"
gt_path = f"gs://allen-nd-goog/ground_truth_tracings/{brain_id}/voxel"
segmentation_path = f"gs://allen-nd-goog/from_google/{brain_id}/whole_brain/{segmentation_id}/"

# Image path (ExaSPIM raw fluorescence)
with open("../configs/exaspim_image_prefixes.json") as f:
    img_prefixes = json.load(f)
img_path = img_prefixes[brain_id]

print(f"Fragments: {fragments_path}")
print(f"Ground truth: {gt_path}")
print(f"Segmentation: {segmentation_path}")
print(f"Image: {img_path}")

Fragments: gs://allen-nd-goog/from_google/794495/whole_brain/raw.unet_449_splits_and_merges_900000/swcs
Ground truth: gs://allen-nd-goog/ground_truth_tracings/794495/voxel
Segmentation: gs://allen-nd-goog/from_google/794495/whole_brain/raw.unet_449_splits_and_merges_900000/
Image: s3://aind-open-data/exaSPIM_794495_2026-01-21_14-25-07_processed_2026-01-29_22-02-08/fusion/fused.zarr/


In [3]:
# Configuration for the proofreading pipeline.
# These parameters control graph construction and proposal generation.

graph_config = ProposalGraphConfig(
    anisotropy=(0.748, 0.748, 1.0),   # ExaSPIM voxel size in microns (x, y, z)
    min_cable_length=40.0,             # Drop fragments shorter than this (microns)
    node_spacing=1.0,                  # Resample spacing between nodes (microns)
    prune_depth=24.0,                  # Prune short terminal branches (microns)
    max_proposals_per_leaf=3,          # Max connection candidates per leaf node
    allow_nonleaf_proposals=False,     # Only propose from leaf (endpoint) nodes
    remove_high_risk_merges=False,
    verbose=True,
)

ml_config = MLConfig(
    batch_size=64,
    brightness_clip=400,               # Clip image intensity for normalization
    device="cuda",                     # Use GPU for inference
    patch_shape=(96, 96, 96),           # 3D image patch around each proposal
    transform=False,                   # No augmentation at inference time
)

config = Config(graph_config, ml_config)
print("Graph config:", graph_config)
print("ML config:", ml_config)

Graph config: ProposalGraphConfig(allow_nonleaf_proposals=False, anisotropy=(0.748, 0.748, 1.0), max_proposals_per_leaf=3, min_cable_length=40.0, node_spacing=1.0, proposals_per_leaf=3, prune_depth=24.0, remove_doubles=True, remove_high_risk_merges=False, trim_endpoints_bool=True, verbose=True)
ML config: MLConfig(batch_size=64, brightness_clip=400, device='cuda', model_name=None, patch_shape=(96, 96, 96), transform=False)


## Train a Model (bounded by epochs or steps)

Trains `VisionHGAT` using `neuron-proofreader`'s own `FragmentsDatasetCollection`
and `Trainer`. The collection builds its own `ProposalGraph`, generates proposals,
and labels them against `gt_path`. Set `max_epochs` and/or `max_steps` to control
how long training runs.

In [ ]:
import torch
from torch.utils.data import DataLoader
from neuron_proofreader.machine_learning.gnn_models import VisionHGAT
from neuron_proofreader.machine_learning.train import Trainer
from neuron_proofreader.split_proofreading.split_datasets import FragmentsDatasetCollection

# Bounded training controls. Set max_steps for a quick smoke run,
# or leave it as None to train complete epochs.
max_epochs = 5
max_steps = 200  # e.g., 200; set to None for full epochs
search_radius = 20.0  # microns

# Use CUDA when available, otherwise fall back to CPU.
ml_config.device = "cuda" if torch.cuda.is_available() else "cpu"
config = Config(graph_config, ml_config)

train_collection = FragmentsDatasetCollection(shuffle=True)
train_collection.add_dataset(
    key=brain_id,
    fragments_path=fragments_path,
    img_path=img_path,
    config=config,
    gt_path=gt_path,
)
train_collection.generate_proposals(search_radius)

# The collection dataset yields model-ready (inputs, targets), so keep
# DataLoader batch_size=None to avoid collating already-batched subgraphs.
train_loader = DataLoader(train_collection, batch_size=None, num_workers=0)
val_loader = DataLoader(train_collection, batch_size=None, num_workers=0)

model = VisionHGAT(patch_shape=ml_config.patch_shape)
trainer = Trainer(
    model=model,
    model_name="VisionHGAT",
    output_dir="training_output",
    device=ml_config.device,
    lr=1e-3,
    max_epochs=max_epochs,
    max_steps=max_steps,
)

trainer.run(train_loader, val_loader)
if trainer.global_step == 0:
    raise RuntimeError("No training steps ran; check proposal generation and GT labels.")

trained_model_path = trainer.latest_checkpoint_path
if trained_model_path is None:
    trained_model_path = trainer.save_model("final")

print(f"Training complete. Checkpoint: {trained_model_path}")
print(f"Training log dir: {trainer.log_dir}")

Filter Doubles: 100%|██████████| 479371/479371 [18:07<00:00, 440.63it/s]


## Run Inference with the Trained Model

The `InferencePipeline` loads the fragments without GT labels, scores proposals
with the trained checkpoint, progressively merges accepted proposals, and saves
the proofreading result.

In [ ]:
from neuron_proofreader.split_proofreading.split_inference import InferencePipeline

# To reuse an existing checkpoint instead of training above, set external_model_path.
external_model_path = None
checkpoint_path = external_model_path or globals().get("trained_model_path")
if checkpoint_path is None:
    raise RuntimeError("Train first or set external_model_path to a .pth checkpoint.")

inference_model = VisionHGAT(patch_shape=ml_config.patch_shape)
inference_model.load_state_dict(torch.load(checkpoint_path, map_location=ml_config.device))
inference_model.to(ml_config.device)
inference_model.eval()

pipeline = InferencePipeline(
    fragments_path=fragments_path,
    img_path=img_path,
    output_dir="proofreader_output",
    model=inference_model,
    config=config,
)

pipeline(
    search_radius=search_radius,
    min_threshold=0.75,
    dt=0.05,
    removal_threshold=0.3,
)

print("Inference complete")
print(f"  Accepted proposals: {len(pipeline.dataset.accepts):,}")
print("  Output saved to: proofreader_output/")

## Before/After Metrics

Quantify which split/merge errors were corrected by comparing the original
fragments against the proofread output.

In [ ]:
from pathlib import Path

proofreader_output = Path("proofreader_output")
print("Proofreading artifacts:")
for rel_path in [
    "corrected_swcs/swcs.zip",
    "connections.txt",
    "runtimes.txt",
    "metadata_graph.json",
    "metadata_ml.json",
]:
    path = proofreader_output / rel_path
    print(f"  {rel_path}: {'OK' if path.exists() else 'missing'}")

# Run before/after skeleton metrics to quantify which errors were corrected.
# This can be slow because it re-reads GT SWCs, fragment SWCs, and the label volume.
import sys
sys.path.insert(0, "../scripts")
from proofreading_stats import evaluate_before_after, print_comparison

metrics_output_dir = f"../metrics_out/{brain_id}/{segmentation_id}/proofreader_before_after"
comparison = evaluate_before_after(
    gt_path=gt_path,
    segmentation_path=segmentation_path,
    original_fragments_path=fragments_path,
    proofreader_output_dir="proofreader_output",
    metrics_output_dir=metrics_output_dir,
    anisotropy=graph_config.anisotropy,
    use_anisotropy=False,
    save_merges=True,
    verbose=True,
)
print_comparison(comparison)
print(f"Before/after metric reports saved to: {metrics_output_dir}")